In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
#installing libraries
!pip install datasets

In [22]:
## From lab 7
## This code here just makes it so you don't need an API
## key for Weights and Biases. Just run it, and you're good.

import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix

os.environ["WANDB_DISABLED"] = "true"

In [23]:
from datasets import load_dataset

csv_file_path = '/content/drive/MyDrive/2025-26/NLP/dancing1000v2.csv'

#load the dataset from the local CSV file
dancing_dataset = load_dataset('csv', data_files=csv_file_path)

#display the first few examples of the dataset
print(dancing_dataset)

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics', 'year', 'genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'artist_ids', 'niche_genres'],
        num_rows: 1000
    })
})


In [24]:
#create a randomized 80/20 train-test split
train_test_dataset = dancing_dataset['train'].train_test_split(test_size=0.2, seed=42)

print(train_test_dataset)

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics', 'year', 'genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'artist_ids', 'niche_genres'],
        num_rows: 800
    })
    test: Dataset({
        features: ['Unnamed: 0', 'id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics', 'year', 'genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'artist_ids', 'niche_genres'],
        num_rows: 200
    })
})


In [25]:
#function to create labels based on danceability score
def create_danceability_labels(example):
    danceability = example['danceability']
    if danceability < 0.4:
        example['labels'] = 0 # 'low danceability'
    elif danceability > 0.6:
        example['labels'] = 1 # 'high danceability'
    else:
        example['labels'] = -1 # Mark for removal or as 'intermediate'
    return example

#apply the labeling function to both train and test splits
train_test_dataset_labeled = train_test_dataset.map(create_danceability_labels)

#filter out examples that are not 'low danceability' or 'high danceability'
train_test_dataset_filtered = train_test_dataset_labeled.filter(lambda example: example['labels'] != -1)

#update the 'train' and 'test' splits with the filtered data
train_dataset = train_test_dataset_filtered['train']
test_dataset = train_test_dataset_filtered['test']

#display the updated datasets to show the new 'labels' column and filtered counts
print("Filtered Training Data:", train_dataset)
print("Filtered Test Data:", test_dataset)

Filtered Training Data: Dataset({
    features: ['Unnamed: 0', 'id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics', 'year', 'genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'artist_ids', 'niche_genres', 'labels'],
    num_rows: 800
})
Filtered Test Data: Dataset({
    features: ['Unnamed: 0', 'id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics', 'year', 'genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'artist_ids', 'niche_genres', 'labels'],
    num_rows: 200
})


In [26]:
!pip install gensim

In [27]:
from gensim.models import Word2Vec

# Extract tokenized words from the training dataset
# The tokenizer function usually returns 'input_ids', which are integer representations of tokens.
# We need to convert these back to words or use the original words if available.
# Assuming `tokenize_function` provides `input_ids`, we'll need to decode them or re-tokenize for Word2Vec.
# For simplicity, let's re-tokenize the raw lyrics into words for Word2Vec training.

# First, get the raw lyrics from the filtered training dataset
raw_train_lyrics = [str(lyric).lower().split() for lyric in train_dataset["lyrics"] if lyric is not None]

# Train Word2Vec model
# Using skip-gram (sg=1), window size of 5, min_count to ignore infrequent words, and 100 dimensions for embeddings
word2vec_model = Word2Vec(sentences=raw_train_lyrics, vector_size=100, window=5, min_count=1, workers=4, sg=1)

print("Word2Vec model training complete.")
print(f"Vocabulary size: {len(word2vec_model.wv)}")

Word2Vec model training complete.
Vocabulary size: 17906


In [28]:
#function to get averaged Word2Vec vector for a document (song lyrics)

import numpy as np

def get_word2vec_embedding(lyrics_tokens, model, vector_size):
    embeddings = []
    for token in lyrics_tokens:
        if token in model.wv:
            embeddings.append(model.wv[token])
    if embeddings:
        return np.mean(embeddings, axis=0)
    else:
        return np.zeros(vector_size)

#create Word2Vec embeddings for train and test datasets
X_train_w2v = np.array([get_word2vec_embedding(str(lyric).lower().split(), word2vec_model, 100)
                        for lyric in train_dataset["lyrics"]])
X_test_w2v = np.array([get_word2vec_embedding(str(lyric).lower().split(), word2vec_model, 100)
                       for lyric in test_dataset["lyrics"]])

y_train = np.array(train_dataset["labels"])
y_test = np.array(test_dataset["labels"])

print(f"Shape of Word2Vec training embeddings: {X_train_w2v.shape}")
print(f"Shape of Word2Vec test embeddings: {X_test_w2v.shape}")

Shape of Word2Vec training embeddings: (800, 100)
Shape of Word2Vec test embeddings: (200, 100)


Now we will train a simple Logistic Regression classifier using these Word2Vec embeddings to predict danceability (low or high).

In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

#training a Logistic Regression classifier
classifier_w2v = LogisticRegression(max_iter=500, random_state=42)
classifier_w2v.fit(X_train_w2v, y_train)

#make predictions on test set
y_pred_w2v = classifier_w2v.predict(X_test_w2v)

#evaluate performance
accuracy_w2v = accuracy_score(y_test, y_pred_w2v)
report_w2v = classification_report(y_test, y_pred_w2v, target_names=["low danceability", "high danceability"])

print(f"Word2Vec Model Accuracy: {accuracy_w2v:.4f}")
print("\nWord2Vec Model Classification Report:")
print(report_w2v)

Word2Vec Model Accuracy: 0.7450

Word2Vec Model Classification Report:
                   precision    recall  f1-score   support

 low danceability       0.75      0.77      0.76       104
high danceability       0.74      0.72      0.73        96

         accuracy                           0.74       200
        macro avg       0.74      0.74      0.74       200
     weighted avg       0.74      0.74      0.74       200

